# Unsloth Studio

Based on https://github.com/unslothai/unsloth/blob/main/studio/Unsloth_Studio_Colab.ipynb -- built-in login, our own cloudflared (quick or fixed domain).

Run it with the service script (`colabllm <name> up`), which creates the Colab session, mounts Drive with `colab dfs`, optionally creates a named Cloudflare tunnel, and injects `CT_*` settings. Cell order is the standard contract -- see `.agents/skills/colab-llm-service/SKILL.md`.

In [ ]:
# @title 0. Config -- values come from `colab exec --env` (defaults allow running in the Colab UI)
import json, os, re, secrets, shutil, stat, subprocess, sys, time, urllib.request, urllib.error

SERVICE      = os.environ.get("CT_SERVICE", "unsloth-studio")
SERVICE_PORT = int(os.environ.get("CT_SERVICE_PORT", "8888"))
AUTH         = os.environ.get("CT_AUTH", "builtin")            # caddy | builtin | none
PUBLIC_PORT  = int(os.environ.get("CT_PUBLIC_PORT", "8888"))  # what cloudflared targets
AUTH_USER    = os.environ.get("CT_AUTH_USER", "admin")
AUTH_PASS    = os.environ.get("CT_AUTH_PASS") or secrets.token_urlsafe(18)
API_TOKEN    = os.environ.get("CT_API_TOKEN") or "sk-" + secrets.token_urlsafe(32)
TUNNEL_TOKEN = os.environ.get("CT_TUNNEL_TOKEN", "")            # set -> fixed domain (named tunnel)
PUBLIC_HOST  = os.environ.get("CT_PUBLIC_HOSTNAME", "")
CREDS_FROM_ENV = bool(os.environ.get("CT_AUTH_PASS"))

WORK = os.environ.get("CT_WORK_DIR", "/content/ct")             # logs + runtime configs (0700)
CACHE_DIR = os.environ.get("CT_CACHE_DIR", "")
BIN_DIR = os.environ.get("CT_BIN_DIR", "/usr/local/bin")
os.makedirs(WORK, mode=0o700, exist_ok=True)
os.chmod(WORK, 0o700)
os.makedirs(BIN_DIR, exist_ok=True)
if BIN_DIR not in os.environ.get("PATH", "").split(os.pathsep):
    os.environ["PATH"] = BIN_DIR + os.pathsep + os.environ.get("PATH", "")
for _p in ("/content/drive/MyDrive/colab-llm/hf_token.txt",
           "/content/drive/MyDrive/colab-services/hf_token.txt",
           "/content/drive/MyDrive/colab-services/qwen-flash/hf_token.txt"):
    if not os.environ.get("HF_TOKEN") and os.path.isfile(_p):
        _tok = open(_p).read().strip()
        if _tok:
            os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = _tok
            break
MOUNT_DRIVE = os.environ.get("CT_MOUNT_DRIVE", "1") != "0"
DATA_DIR = os.environ.get("CT_DATA_DIR", f"/content/drive/MyDrive/colab-llm/{SERVICE}")
if not MOUNT_DRIVE or not os.path.isdir("/content/drive/MyDrive"):   # Drive not mounted -> local fallback
    if DATA_DIR.startswith("/content/drive/"):
        DATA_DIR = f"/content/data/{SERVICE}"
os.makedirs(DATA_DIR, exist_ok=True)
PROCS = {}                                                      # name -> (Popen, restart function)

def ct_spawn(name, args, env=None, cwd=None):
    """Start a background process; output goes to <WORK>/<name>.log (0600)."""
    lp = f"{WORK}/{name}.log"
    fd = os.open(lp, os.O_WRONLY | os.O_CREAT | os.O_APPEND, 0o600)
    os.chmod(lp, 0o600)
    with os.fdopen(fd, "a") as logf:   # child keeps its own dup of the fd
        return subprocess.Popen(args, stdout=logf, stderr=subprocess.STDOUT, cwd=cwd,
                                env={**os.environ, **(env or {})}, start_new_session=True)

CADDY_VERSION = "2.10.2"

def ct_write_private_text(path, content):
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, "w", encoding="utf-8") as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())

def ct_proc_start_time(pid):
    try:
        fields = open(f"/proc/{int(pid)}/stat", "r", encoding="utf-8", errors="replace").read().rsplit(")", 1)[1].split()
        if fields[0] == "Z":
            return None
        return fields[19]
    except (OSError, IndexError, ValueError):
        return None

def ct_stop_marked_proc(name, timeout=10):
    import signal
    marker = f"{WORK}/{name}.proc.json"
    if not os.path.exists(marker):
        return
    try:
        info = json.loads(open(marker, "r", encoding="utf-8").read())
        pid = int(info.get("pid") or 0)
        start_time = info.get("start_time")
    except (OSError, ValueError, TypeError):
        pid, start_time = 0, None
    if pid > 0 and start_time and ct_proc_start_time(pid) == start_time:
        for sig in (signal.SIGTERM, signal.SIGKILL):
            if ct_proc_start_time(pid) != start_time:
                break
            try:
                os.killpg(os.getpgid(pid), sig)
            except (ProcessLookupError, PermissionError, OSError):
                try:
                    os.kill(pid, sig)
                except (ProcessLookupError, OSError):
                    break
            deadline = time.monotonic() + timeout
            while time.monotonic() < deadline:
                if ct_proc_start_time(pid) != start_time:
                    break
                time.sleep(0.1)
    try:
        os.unlink(marker)
    except OSError:
        pass

def ct_spawn_marked(name, args, env=None, cwd=None):
    ct_stop_marked_proc(name)
    proc = ct_spawn(name, args, env=env, cwd=cwd)
    marker = f"{WORK}/{name}.proc.json"
    ct_write_private_text(marker, json.dumps({"pid": proc.pid, "start_time": ct_proc_start_time(proc.pid)}))
    return proc

def _ct_file_sha256(path):
    import hashlib
    h = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def ct_ensure_caddy():
    import hashlib, platform, tarfile, tempfile
    arch = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine(), "amd64")
    dest = f"{BIN_DIR}/caddy"
    status_label = "miss"
    manifest = f"{CACHE_DIR}/manifests/caddy.json" if CACHE_DIR else ""
    if os.path.exists(dest):
        if not manifest:
            return dest
        try:
            meta = json.loads(open(manifest, "r", encoding="utf-8").read())
            if meta.get("artifact") == "caddy" and meta.get("version") == CADDY_VERSION and meta.get("sha256") == _ct_file_sha256(dest):
                print("CT_CACHE_EVENT artifact=caddy status=hit", flush=True)
                return dest
        except (OSError, ValueError):
            pass
        try:
            os.unlink(dest)
        except OSError:
            pass
        status_label = "corrupt_rebuilt"
    asset = f"caddy_{CADDY_VERSION}_linux_{arch}.tar.gz"
    checks_asset = f"caddy_{CADDY_VERSION}_checksums.txt"
    base = f"https://github.com/caddyserver/caddy/releases/download/v{CADDY_VERSION}/"
    os.makedirs(BIN_DIR, exist_ok=True)
    with tempfile.TemporaryDirectory() as tmp:
        tgz = os.path.join(tmp, asset)
        with urllib.request.urlopen(base + asset, timeout=60) as src, open(tgz, "wb") as dst:
            shutil.copyfileobj(src, dst)
        checks = urllib.request.urlopen(base + checks_asset, timeout=60).read().decode("utf-8")
        expected = next((line.split()[0] for line in checks.splitlines() if line.split() and line.split()[-1].lstrip("*") == asset), None)
        algo = {64: hashlib.sha256, 128: hashlib.sha512}.get(len(expected or ""))
        if algo is None or algo(open(tgz, "rb").read()).hexdigest() != expected:
            raise RuntimeError(f"Caddy checksum verification failed for {asset}")
        partial = f"{dest}.{os.getpid()}.partial"
        try:
            with tarfile.open(tgz) as tar:
                entry = tar.getmember("caddy")
                with tar.extractfile(entry) as src, open(partial, "wb") as out:
                    shutil.copyfileobj(src, out)
            os.chmod(partial, 0o755)
            os.replace(partial, dest)
        finally:
            if os.path.exists(partial):
                os.unlink(partial)
    if manifest:
        os.makedirs(os.path.dirname(manifest), exist_ok=True)
        tmp_m = f"{manifest}.{os.getpid()}.tmp"
        with open(tmp_m, "w", encoding="utf-8") as mf:
            mf.write(json.dumps({
                "artifact": "caddy",
                "version": CADDY_VERSION,
                "sha256": _ct_file_sha256(dest),
                "bin_path": dest,
                "verified_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            }, indent=2) + "\n")
        os.replace(tmp_m, manifest)
        print(f"CT_CACHE_EVENT artifact=caddy status={status_label}", flush=True)
    return dest

def ct_stop_proc(proc, timeout=15):
    """Terminate a ct_spawn process group, escalating to SIGKILL after a bounded wait."""
    import signal
    if proc is None or proc.poll() is not None:
        return
    for sig in (signal.SIGTERM, signal.SIGKILL):
        try:
            os.killpg(proc.pid, sig)
        except (ProcessLookupError, PermissionError):
            try:
                proc.send_signal(sig)
            except ProcessLookupError:
                return
        try:
            proc.wait(timeout=timeout)
            return
        except subprocess.TimeoutExpired:
            continue

def ct_gpu_stat(query="memory.used,memory.total,utilization.gpu"):
    """nvidia-smi summary that never raises (CPU runtimes / transient driver errors)."""
    try:
        return subprocess.check_output(["nvidia-smi", f"--query-gpu={query}", "--format=csv,noheader"],
                                       text=True, stderr=subprocess.DEVNULL, timeout=20).strip()
    except (OSError, subprocess.SubprocessError):
        return "cpu/no-gpu"

def ct_log_size(name):
    p = f"{WORK}/{name}.log"
    return os.path.getsize(p) if os.path.exists(p) else 0

def ct_log_since(name, offset):
    p = f"{WORK}/{name}.log"
    if not os.path.exists(p):
        return ""
    with open(p, "rb") as handle:
        handle.seek(min(offset, os.path.getsize(p)))
        return handle.read().decode("utf-8", errors="replace")

def ct_public_gate(public_url, probe, verify_auth, log, timeout=120, interval=8):
    """Wait for the public endpoint; auth failures are fatal, network errors are retried."""
    deadline = time.monotonic() + timeout
    while time.monotonic() < deadline:
        try:
            code = probe()
        except Exception as err:   # DNS / edge propagation / TLS warm-up
            log(f"Public probe pending ({type(err).__name__}: {err})")
            time.sleep(interval)
            continue
        if code == 200:
            verify_auth()            # RuntimeError propagates: never report CT_READY with broken auth
            return
        log(f"Public probe pending (HTTP {code})")
        time.sleep(interval)
    raise RuntimeError(f"Public endpoint {public_url} failed readiness verification within {timeout}s")

def ct_watchdog(health, start_service, log, *, service_key="service", log_name="service",
                interval=60, grace=900, dead_markers=(), heartbeat=None, heartbeat_interval=None):
    """Supervise PROCS forever: restart dead processes, debounce health failures after reloads."""
    started = time.monotonic()
    last_heartbeat = started
    failures = 0
    grace_until = time.monotonic()
    log_offset = ct_log_size(log_name)

    def restart_service(reason):
        nonlocal failures, grace_until, log_offset
        log(f"[watchdog] {reason}; restarting {service_key}")
        ct_stop_proc(PROCS[service_key][0])
        PROCS[service_key] = (start_service(), start_service)
        failures = 0
        grace_until = time.monotonic() + grace
        log_offset = ct_log_size(log_name)

    while True:
        time.sleep(interval)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                log(f"[watchdog] {name} exited with {proc.returncode}; restarting\n{ct_tail(name, 20)}")
                PROCS[name] = (restart(), restart)
                if name == service_key:
                    failures = 0
                    grace_until = time.monotonic() + grace
                    log_offset = ct_log_size(log_name)
        if dead_markers:
            fresh = ct_log_since(log_name, log_offset)
            log_offset += len(fresh.encode("utf-8", errors="replace"))
            if any(marker in fresh for marker in dead_markers):
                restart_service("Detected dead worker marker in service log")
                continue
        try:
            healthy = bool(health())
        except Exception:
            healthy = False
        if healthy:
            failures = 0
        elif time.monotonic() >= grace_until:
            failures += 1
        if failures >= 6:
            restart_service("6 consecutive failed health checks")
        now_m = time.monotonic()
        if heartbeat is not None and (heartbeat_interval is None or (now_m - last_heartbeat) >= heartbeat_interval):
            last_heartbeat = now_m
            try:
                heartbeat(int(now_m - started))
            except Exception as err:
                log(f"[watchdog] heartbeat warning: {err}")

def ct_wait_http(url, timeout=600, ok=lambda code: code < 500):
    """Wait until url answers with a non-5xx status (401/404 count as 'listening')."""
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if ok(r.status): return r.status
        except urllib.error.HTTPError as e:
            if ok(e.code): return e.code
        except Exception:
            pass
        time.sleep(2)
    raise TimeoutError(f"{url} not ready after {timeout}s")

def ct_tail(name, n=30):
    p = f"{WORK}/{name}.log"
    return "".join(open(p, errors="replace").readlines()[-n:]) if os.path.exists(p) else ""

def ct_fetch_binary(path, url):
    if not os.path.exists(path):
        partial = f"{path}.{os.getpid()}.partial"
        os.makedirs(os.path.dirname(path), exist_ok=True)
        with urllib.request.urlopen(url, timeout=60) as src, open(partial, "wb") as dst:
            shutil.copyfileobj(src, dst)
        os.chmod(partial, os.stat(partial).st_mode | stat.S_IEXEC)
        os.replace(partial, path)
    return path

print(f"service={SERVICE} port={SERVICE_PORT} auth={AUTH} public_port={PUBLIC_PORT} "
      f"tunnel={'named ' + PUBLIC_HOST if TUNNEL_TOKEN else 'quick'} data={DATA_DIR}")


In [ ]:
# @title 1. Setup -- clone or update the repo, then Unsloth's own installer (re-runs are fast)
%cd /content
!test -d unsloth/.git && git -C unsloth pull --ff-only || git clone --depth 1 --branch main https://github.com/unslothai/unsloth.git
%cd /content/unsloth
!chmod +x studio/setup.sh && ./studio/setup.sh --local


In [ ]:
# @title 2. Service -- Unsloth Studio is started by its own blocking start() in cell 5;
# this cell only prepares the import path. It has built-in auth (CT_AUTH=builtin), so no Caddy.
sys.path.insert(0, "/content/unsloth/studio/backend")
print("Unsloth Studio will listen on port", SERVICE_PORT)


In [ ]:
# @title 3. Auth layer -- Caddy: basic auth for browsers, `Authorization: Bearer <token>` for API clients
# PUBLIC_PORT must be free on the VM: Colab itself uses 8080 (runtime server) and 8012 (colab dfs shim).
# Only used when the service has no auth of its own (CT_AUTH=caddy). The service itself listens on
# 127.0.0.1 only, cloudflared targets Caddy, so every public request passes through auth.
REWRITE_HOST = True   # True: send Host: 127.0.0.1:<port> upstream (Ollama & co. reject foreign Host headers)

def start_caddy():
    caddy_bin = ct_ensure_caddy()
    import hashlib
    cookie_tok = hashlib.sha256(f"ct-cookie-v2:{AUTH_USER}:{AUTH_PASS}:{API_TOKEN}".encode()).hexdigest()
    hashed = subprocess.check_output([caddy_bin, "hash-password", "--plaintext", AUTH_PASS], text=True).strip()
    upstream = f"""reverse_proxy 127.0.0.1:{SERVICE_PORT} {{
            {"header_up Host {upstream_hostport}" if REWRITE_HOST else ""}
            {"header_up Origin http://{upstream_hostport}" if REWRITE_HOST else ""}
            header_up -Authorization
            flush_interval -1
        }}"""
    caddyfile = f"""{{
    admin off
    auto_https off
}}
:{PUBLIC_PORT} {{
    encode zstd gzip
    @bearer header Authorization "Bearer {API_TOKEN}"
    handle @bearer {{
        {upstream}
    }}
    @cookie header Cookie *ct_session={cookie_tok}*
    handle @cookie {{
        {upstream}
    }}
    handle {{
        route {{
            basic_auth {{
                "{AUTH_USER}" {hashed}
            }}
            header +Set-Cookie "ct_session={cookie_tok}; Path=/; HttpOnly; SameSite=Lax"
            {upstream}
        }}
    }}
}}
"""
    ct_write_private_text(f"{WORK}/Caddyfile", caddyfile)
    return ct_spawn_marked("caddy", [caddy_bin, "run", "--config", f"{WORK}/Caddyfile", "--adapter", "caddyfile"])

AUTH_OK = False   # cell 4 refuses to open the tunnel unless this becomes True (fail closed)
if AUTH == "caddy":
    PROCS["caddy"] = (start_caddy(), start_caddy)
    time.sleep(2)
    if PROCS["caddy"][0].poll() is not None:   # e.g. port taken (Colab uses 8080 itself)
        raise RuntimeError(f"caddy exited -- is :{PUBLIC_PORT} in use?\n" + ct_tail("caddy"))
    # Must be 401 from *our* Caddy and must NOT leak Set-Cookie on unauthenticated 401 responses.
    ct_wait_http(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=60, ok=lambda c: c == 401)
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PUBLIC_PORT}/", timeout=5)
    except urllib.error.HTTPError as _e:
        if _e.headers.get("Set-Cookie"):
            raise RuntimeError("Caddy leaked Set-Cookie on unauthenticated 401 response!")
    AUTH_OK = True
    print(f"caddy on :{PUBLIC_PORT} -> 127.0.0.1:{SERVICE_PORT} (unauthenticated request -> 401)")
else:
    AUTH_OK = True
    print(f"auth={AUTH}: no proxy, cloudflared targets the service on :{PUBLIC_PORT} directly")


In [ ]:
# @title 4. Tunnel -- Cloudflare by default, or project-configured FRP + HTTPS
"""Self-contained Colab transport cell; injected into every service notebook."""
import hashlib
import json
import os
import platform
import re
import shutil
import signal
import subprocess
import tarfile
import tempfile
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

CLOUDFLARED_VERSION = '2026.11.0'
CLOUDFLARED_SHA256 = {
    'amd64': '51f7c68b57ed7269164075d7e789efce5b59355982f06392deeb037d982a51fd',
    'arm64': 'abe54a006acf9faf2d19ee9d953cb0d0804e92f59c5382a162e57b84d086f56f',
}


def _write_private_text(path, content):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    with os.fdopen(fd, 'w', encoding='utf-8') as handle:
        handle.write(content)
        handle.flush()
        os.fsync(handle.fileno())


def _touch_private_log(path):
    path = Path(path)
    fd = os.open(path, os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
    os.chmod(path, 0o600)
    os.close(fd)


def _file_sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            chunk = handle.read(65536)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def _check_cached_binary(destination, artifact, version, cache_dir):
    if not destination.exists():
        return None
    if not cache_dir:
        return 'hit'
    manifest = Path(cache_dir) / 'manifests' / f'{artifact}.json'
    if not manifest.is_file():
        return 'miss'
    try:
        meta = json.loads(manifest.read_text(encoding='utf-8'))
        if (
            meta.get('artifact') == artifact
            and meta.get('version') == version
            and meta.get('sha256') == _file_sha256(destination)
        ):
            return 'hit'
    except (OSError, ValueError):
        pass
    destination.unlink(missing_ok=True)
    return 'corrupt_rebuilt'


def _write_manifest(cache_dir, artifact, version, destination):
    if not cache_dir:
        return
    manifest_dir = Path(cache_dir) / 'manifests'
    manifest_dir.mkdir(parents=True, exist_ok=True)
    manifest = manifest_dir / f'{artifact}.json'
    tmp = manifest.with_name(f'.{artifact}.{os.getpid()}.tmp')
    payload = {
        'artifact': artifact,
        'version': version,
        'sha256': _file_sha256(destination),
        'bin_path': str(destination),
        'verified_at': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()),
    }
    tmp.write_text(json.dumps(payload, indent=2) + '\n', encoding='utf-8')
    os.replace(tmp, manifest)


def render_frpc(config, local_port, ca_path):
    """TOML strings use JSON quoting, never shell interpolation."""
    return f'''serverAddr = {json.dumps(config['server_ip'])}
serverPort = {int(config['server_port'])}
loginFailExit = false
auth.method = "token"
auth.token = {json.dumps(config['token'])}
auth.additionalScopes = {json.dumps(config.get('auth_scopes',['HeartBeats','NewWorkConns']))}
transport.tls.enable = true
transport.tls.trustedCaFile = {json.dumps(str(ca_path))}
transport.tls.serverName = {json.dumps(config.get('server_name') or config['server_ip'])}
log.to = "console"
log.level = "info"
[[proxies]]
name = {json.dumps(config['proxy_name'])}
type = "tcp"
localIP = "127.0.0.1"
localPort = {int(local_port)}
remotePort = {int(config['remote_port'])}
'''


def install_frpc(destination, version, *, cache_dir=None):
    destination = Path(destination)
    cache_state = _check_cached_binary(destination, 'frpc', version, cache_dir)
    if cache_state == 'hit':
        if cache_dir:
            print(f'CT_CACHE_EVENT artifact=frpc status=hit', flush=True)
        return str(destination)
    status_label = cache_state or 'miss'
    arch={'x86_64':'amd64','aarch64':'arm64'}.get(platform.machine())
    if not arch:
        raise ValueError('Unsupported FRP client architecture')
    if not re.fullmatch(r'\d+\.\d+\.\d+',version) or tuple(map(int,version.split('.')))<(0,61,0):
        raise ValueError('Unsupported FRP client version; expected stable >= 0.61.0')
    asset=f'frp_{version}_linux_{arch}.tar.gz'
    base=f'https://github.com/fatedier/frp/releases/download/v{version}/'
    with tempfile.TemporaryDirectory() as tmp:
        archive=Path(tmp)/asset
        with urllib.request.urlopen(base+asset,timeout=60) as response, open(archive,'wb') as out:
            shutil.copyfileobj(response,out)
        checks=urllib.request.urlopen(base+'frp_sha256_checksums.txt',timeout=60).read().decode()
        expected=next((line.split()[0] for line in checks.splitlines()
                       if line.split() and line.split()[-1].lstrip('*')==asset),None)
        if expected is None or hashlib.sha256(archive.read_bytes()).hexdigest()!=expected:
            raise ValueError('FRP client checksum verification failed')
        with tarfile.open(archive) as tar:
            entry=tar.getmember(f'frp_{version}_linux_{arch}/frpc')
            if not entry.isfile():
                raise ValueError('Invalid FRP client binary')
            destination.parent.mkdir(parents=True,exist_ok=True)
            partial=destination.with_name(f'{destination.name}.{os.getpid()}.partial')
            try:
                with tar.extractfile(entry) as source, open(partial, 'wb') as out_bin:
                    shutil.copyfileobj(source, out_bin)
                partial.chmod(0o755)
                os.replace(partial, destination)
            finally:
                partial.unlink(missing_ok=True)
    _write_manifest(cache_dir, 'frpc', version, destination)
    if cache_dir:
        print(f'CT_CACHE_EVENT artifact=frpc status={status_label}', flush=True)
    return str(destination)


def install_cloudflared(destination, *, cache_dir=None, fetch_fn=None):
    destination = Path(destination)
    cf_arch = {'x86_64': 'amd64', 'aarch64': 'arm64'}.get(platform.machine(), 'amd64')
    url = f'https://github.com/cloudflare/cloudflared/releases/download/{CLOUDFLARED_VERSION}/cloudflared-linux-{cf_arch}'
    expected_sha = CLOUDFLARED_SHA256.get(cf_arch)
    if not expected_sha:
        raise ValueError(f'Unsupported cloudflared architecture: {platform.machine()}')
    if cache_dir:
        cache_state = _check_cached_binary(destination, 'cloudflared', CLOUDFLARED_VERSION, cache_dir)
        if cache_state == 'hit':
            print('CT_CACHE_EVENT artifact=cloudflared status=hit', flush=True)
            return str(destination)
        status_label = cache_state or 'miss'
        if destination.exists() and _file_sha256(destination) != expected_sha:
            destination.unlink(missing_ok=True)
            status_label = 'corrupt_rebuilt'
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
        _write_manifest(cache_dir, 'cloudflared', CLOUDFLARED_VERSION, destination)
        print(f'CT_CACHE_EVENT artifact=cloudflared status={status_label}', flush=True)
        return str(destination)
    if fetch_fn is not None:
        return fetch_fn(str(destination), url)
    if not destination.exists():
        destination.parent.mkdir(parents=True, exist_ok=True)
        partial = destination.with_name(f'{destination.name}.{os.getpid()}.partial')
        try:
            with urllib.request.urlopen(url, timeout=60) as src, open(partial, 'wb') as dst:
                shutil.copyfileobj(src, dst)
            if _file_sha256(partial) != expected_sha:
                raise ValueError('cloudflared checksum verification failed')
            partial.chmod(0o755)
            os.replace(partial, destination)
        finally:
            partial.unlink(missing_ok=True)
    return str(destination)


def run_tunnel(ctx):
    if not ctx.get('AUTH_OK'):
        raise RuntimeError('Auth layer not ready; refusing to expose the service')
    work=Path(ctx.get('WORK') or os.environ.get('CT_WORK_DIR') or '/content/ct')
    work.mkdir(parents=True, exist_ok=True)
    bin_dir_explicit = ctx.get('BIN_DIR') or os.environ.get('CT_BIN_DIR') or ''
    bin_dir = Path(bin_dir_explicit or '/usr/local/bin')
    cache_dir = ctx.get('CACHE_DIR') or os.environ.get('CT_CACHE_DIR') or ''
    transport=os.environ.get('CT_TRANSPORT','cloudflare')
    if transport not in ('cloudflare','frp'):
        raise ValueError('Unsupported tunnel transport')
    # Stop only the tunnel previously started by this notebook (PID + proc
    # start time prevent accidental termination after PID reuse).
    marker=work/'tunnel-process.json'
    def identity(pid):
        try:
            stat=Path(f'/proc/{pid}/stat').read_text().rsplit(')',1)[1].split()
            return stat[19]
        except OSError:
            return None
    if ctx.get('ct_stop_marked_proc') is not None:
        stop_fn = ctx['ct_stop_marked_proc']
        stop_fn('tunnel-process')
    elif marker.exists():
        previous=json.loads(marker.read_text())
        if previous.get('start_time') and identity(previous['pid'])==previous['start_time']:
            os.kill(previous['pid'],signal.SIGTERM)
            for _ in range(50):
                if identity(previous['pid']) is None:
                    break
                time.sleep(.1)
    def spawn(name,args):
        _touch_private_log(work/(name+'.log'))
        proc=ctx['ct_spawn'](name,args)
        _write_private_text(marker, json.dumps(dict(pid=proc.pid,start_time=identity(proc.pid))))
        return proc

    if transport=='frp':
        config={key:os.environ['CT_FRP_'+key.upper()] for key in
                ('server_ip','server_port','remote_port','proxy_name','token')}
        config['server_name']=os.environ.get('CT_FRP_SERVER_NAME') or config['server_ip']
        config['auth_scopes']=json.loads(os.environ.get('CT_FRP_AUTH_SCOPES') or '["HeartBeats","NewWorkConns"]')
        public_host=ctx['PUBLIC_HOST']
        if not public_host:
            raise ValueError('FRP requires a public hostname')
        ca_path=work/'frp-ca.crt'
        _write_private_text(ca_path, os.environ['CT_FRP_CA_CERT'])
        config_path=work/'frpc.toml'
        _write_private_text(config_path, render_frpc(config,ctx['PUBLIC_PORT'],ca_path))
        frpc_name = f"frpc-{os.environ['CT_FRP_VERSION']}"
        frpc_dest = (bin_dir / frpc_name) if bin_dir_explicit else ((work / frpc_name) if (work / frpc_name).exists() else (bin_dir / frpc_name))
        binary=install_frpc(frpc_dest, os.environ['CT_FRP_VERSION'], cache_dir=cache_dir or None)
        result=subprocess.run([binary,'verify','-c',str(config_path)],capture_output=True)
        if result.returncode:
            raise RuntimeError('FRP client configuration validation failed')
        def start():
            return spawn('frpc',[binary,'-c',str(config_path)])
        key='frpc'
    else:
        binary=install_cloudflared(bin_dir / 'cloudflared', cache_dir=cache_dir or None, fetch_fn=ctx.get('ct_fetch_binary'))
        def refresh_quick_url(proc_ref, wait_seconds=60):
            deadline_q=time.monotonic()+wait_seconds
            while time.monotonic()<deadline_q:
                if proc_ref.poll() is not None:
                    return None
                match_q=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',ctx['ct_tail']('cloudflared',100))
                if match_q:
                    new_url=match_q.group(0)
                    if new_url!=ctx.get('PUBLIC_URL'):
                        ctx['PUBLIC_URL']=new_url
                        print(f'CT_PUBLIC_URL={new_url}',flush=True)
                    return new_url
                time.sleep(1)
            return None
        ctx['ct_refresh_public_url']=refresh_quick_url
        def start():
            args=[binary,'tunnel','--no-autoupdate','--protocol','http2']
            if ctx['TUNNEL_TOKEN']:
                args+=['run','--token',ctx['TUNNEL_TOKEN']]
            else:
                args+=['--url',f'http://127.0.0.1:{ctx["PUBLIC_PORT"]}']
            proc_new=spawn('cloudflared',args)
            if not ctx['TUNNEL_TOKEN'] and ctx.get('PUBLIC_URL'):
                threading.Thread(target=refresh_quick_url,args=(proc_new,),daemon=True).start()
            return proc_new
        key='cloudflared'
    proc=start()
    ctx['PROCS'][key]=(proc,start)
    ctx['TUNNEL_PROCESS_KEY']=key
    deadline=time.monotonic()+180
    registered=False
    while time.monotonic()<deadline:
        if proc.poll() is not None:
            raise RuntimeError(f'{key} exited before registration; see {work}/{key}.log')
        log=ctx['ct_tail'](key,100)
        if transport=='frp':
            registered=registered or ('start proxy success' in log and config['proxy_name'] in log)
            if registered:
                url='https://'+ctx['PUBLIC_HOST']
                try:
                    with urllib.request.urlopen(url,timeout=5) as response:
                        if response.status<600:
                            break
                except urllib.error.HTTPError as error:
                    # A valid public TLS response proves the transport/site.
                    # 502 is expected while a large model is still loading;
                    # the service's keep-alive cell gates CT_READY on health.
                    code=error.code
                    error.close()
                    if code<600:
                        break
                except OSError:
                    pass
        elif ctx['TUNNEL_TOKEN'] and 'Registered tunnel connection' in log:
            url='https://'+ctx['PUBLIC_HOST']
            break
        elif not ctx['TUNNEL_TOKEN']:
            match=re.search(r'https://[a-z0-9-]+\.trycloudflare\.com',log)
            if match:
                url=match.group(0)
                break
        time.sleep(1)
    else:
        raise RuntimeError(f'{key} not ready; check tunnel logs, DNS, HTTPS and server ports')
    ctx['PUBLIC_URL']=url
    print(f'CT_PUBLIC_URL={url}',flush=True)

run_tunnel(globals())


In [ ]:
# @title 5. Ready + keep-alive -- KEEP THIS CELL RUNNING
# start() blocks forever (its own keep-alive loop). Our tunnel from cell 4 is used, so cloudflare=False.
# A watcher thread ensures admin credentials + CT_READY and restarts cloudflared if it dies.
import pathlib
import threading

pathlib.Path(f"{WORK}/unsloth_auth_helper.py").write_text(
    '"""Unsloth Studio first-run password reconciliation helper for Colab notebooks."""\n\nfrom __future__ import annotations\n\nimport json\nimport os\nimport time\nimport urllib.error\nimport urllib.request\n\n_BOOTSTRAP_FILES = (".bootstrap_password", ".colab_notebook_login")\n\n\ndef _http_json(\n    url: str,\n    *,\n    method: str = "GET",\n    payload: dict[str, object] | None = None,\n    headers: dict[str, str] | None = None,\n    timeout: float = 15.0,\n    opener=None,\n) -> tuple[int, dict[str, object]]:\n    req_headers = {"Accept": "application/json"}\n    if headers:\n        req_headers.update(headers)\n    data = None\n    if payload is not None:\n        data = json.dumps(payload).encode("utf-8")\n        req_headers["Content-Type"] = "application/json"\n    req = urllib.request.Request(url, data=data, headers=req_headers, method=method)\n    open_fn = opener or urllib.request.urlopen\n    try:\n        with open_fn(req, timeout=timeout) as resp:\n            status = getattr(resp, "status", None) or resp.getcode()\n            raw = resp.read().decode("utf-8", "replace")\n    except urllib.error.HTTPError as err:\n        raw = err.read().decode("utf-8", "replace")\n        try:\n            body = json.loads(raw) if raw.strip() else {}\n        except ValueError:\n            body = {"detail": raw.strip()}\n        return err.code, body if isinstance(body, dict) else {"detail": body}\n    try:\n        body = json.loads(raw) if raw.strip() else {}\n    except ValueError:\n        body = {}\n    return status, body if isinstance(body, dict) else {}\n\n\ndef _read_candidate_passwords(auth_dir: str) -> list[str]:\n    candidates: list[str] = []\n    for filename in _BOOTSTRAP_FILES:\n        path = os.path.join(auth_dir, filename)\n        if not os.path.isfile(path):\n            continue\n        try:\n            tokens = open(path, encoding="utf-8", errors="replace").read().split()\n        except OSError:\n            continue\n        pw = tokens[-1] if tokens else ""\n        if pw and pw not in candidates:\n            candidates.append(pw)\n    return candidates\n\n\ndef _login(\n    base_url: str,\n    username: str,\n    password: str,\n    *,\n    opener=None,\n) -> tuple[int, dict[str, object]]:\n    return _http_json(\n        f"{base_url}/api/auth/login",\n        method="POST",\n        payload={"username": username, "password": password},\n        opener=opener,\n    )\n\n\ndef _change_password(\n    base_url: str,\n    access_token: str,\n    current_password: str,\n    new_password: str,\n    *,\n    opener=None,\n) -> tuple[int, dict[str, object]]:\n    return _http_json(\n        f"{base_url}/api/auth/change-password",\n        method="POST",\n        payload={\n            "current_password": current_password,\n            "new_password": new_password,\n        },\n        headers={"Authorization": f"Bearer {access_token}"},\n        opener=opener,\n    )\n\n\ndef ensure_unsloth_admin_password(\n    base_url: str,\n    target_password: str,\n    *,\n    auth_dir: str = "/root/.unsloth/studio/auth",\n    prefer_target: bool = True,\n    timeout: float = 120.0,\n    poll_interval: float = 1.5,\n    opener=None,\n) -> tuple[str, str]:\n    """Ensure Unsloth Studio\'s admin password matches target_password when feasible.\n\n    Returns ``(username, active_password)`` where ``active_password`` is confirmed\n    to log in against ``/api/auth/login`` (or the discovered bootstrap password if\n    changing password is not possible).\n    """\n    base = base_url.rstrip("/")\n    deadline = time.monotonic() + timeout\n    username = "unsloth"\n    last_login_pw = ""\n\n    while True:\n        requires_change = False\n        try:\n            code, status = _http_json(f"{base}/api/auth/status", opener=opener)\n        except OSError:\n            code, status = 0, {}\n\n        if code == 200 and status.get("initialized", True):\n            username = str(status.get("default_username") or "unsloth")\n            requires_change = bool(status.get("requires_password_change"))\n\n            if prefer_target and target_password:\n                login_code, login_body = _login(\n                    base, username, target_password, opener=opener\n                )\n                if login_code == 200:\n                    if not (\n                        requires_change or login_body.get("must_change_password")\n                    ):\n                        return username, target_password\n                    token = str(login_body.get("access_token") or "")\n                    if token and len(target_password) >= 8:\n                        chg_code, _ = _change_password(\n                            base,\n                            token,\n                            target_password,\n                            target_password,\n                            opener=opener,\n                        )\n                        if chg_code == 200:\n                            return username, target_password\n                    return username, target_password\n\n            for cand in _read_candidate_passwords(auth_dir):\n                login_code, login_body = _login(base, username, cand, opener=opener)\n                if login_code != 200:\n                    continue\n                last_login_pw = cand\n                token = str(login_body.get("access_token") or "")\n                if (\n                    target_password\n                    and len(target_password) >= 8\n                    and cand != target_password\n                    and token\n                ):\n                    chg_code, chg_body = _change_password(\n                        base, token, cand, target_password, opener=opener\n                    )\n                    if chg_code == 200:\n                        try:\n                            _, post_status = _http_json(\n                                f"{base}/api/auth/status", opener=opener\n                            )\n                        except OSError:\n                            post_status = {}\n                        if not post_status.get("requires_password_change"):\n                            return username, target_password\n                        verify_code, _ = _login(\n                            base, username, target_password, opener=opener\n                        )\n                        if verify_code == 200:\n                            return username, target_password\n                    else:\n                        detail = chg_body.get("detail") or chg_body\n                        print(\n                            f"[unsloth-auth] change-password HTTP {chg_code}: {detail}",\n                            flush=True,\n                        )\n                elif not (requires_change or login_body.get("must_change_password")):\n                    return username, cand\n\n        if time.monotonic() >= deadline:\n            if last_login_pw:\n                return username, last_login_pw\n            candidates = _read_candidate_passwords(auth_dir)\n            if candidates:\n                return username, candidates[0]\n            return username, ""\n\n        time.sleep(poll_interval)\n',
    encoding="utf-8",
)
if WORK not in sys.path:
    sys.path.insert(0, WORK)
from unsloth_auth_helper import ensure_unsloth_admin_password

def _watch():
    try:
        _watch_inner()
    except BaseException as err:   # a dead watcher would otherwise leave the CLI waiting forever
        import traceback
        traceback.print_exc()
        print(f"[watchdog] fatal: {err}", flush=True)
        os._exit(1)

def _watch_inner():
    ct_wait_http(f"http://127.0.0.1:{SERVICE_PORT}/api/health", timeout=900)
    user, pw = ensure_unsloth_admin_password(
        f"http://127.0.0.1:{SERVICE_PORT}",
        AUTH_PASS,
        prefer_target=CREDS_FROM_ENV,
    )
    if pw:
        print(f"CT_CREDENTIALS user={user} pass={pw}", flush=True)
    print(f"{SERVICE} is up: {PUBLIC_URL}")
    print("CT_READY", flush=True)
    last_beat = time.time()
    while True:
        time.sleep(30)
        for name, (proc, restart) in list(PROCS.items()):
            if proc.poll() is not None:
                print(f"[watchdog] {name} exited; restarting", flush=True)
                ct_stop_proc(proc)
                PROCS[name] = (restart(), restart)
        if time.time() - last_beat > 240:
            last_beat = time.time()
            states = " ".join(f"{n}={'up' if p.poll() is None else 'down'}" for n, (p, _) in PROCS.items())
            print(f"[heartbeat] {time.strftime('%H:%M:%S')} {states} gpu={ct_gpu_stat()}", flush=True)

threading.Thread(target=_watch, daemon=True).start()
from colab import start
start(port=SERVICE_PORT, cloudflare=False)
